In [ ]:
import numpy as np
from scipy.signal import hilbert
from joblib import Parallel, delayed
import os
import mne
import DeepNeuralNetworkSleep.hdf5_files.Artefacts_Detection as A
import Sleep_Scripts.Cleaned_feature_vis as F
import matplotlib.pyplot as plt
import pickle

import pandas as pd


In [ ]:
noise_band = [0,0.5]
delta_band = [0.5,5]
theta_band = [6,10]
sigma_band = [11,17]
beta_band = [22,30]
gamma_band = [35,45]
total_band = [0,30]
alpha_band = [8,13]

### Functions

In [ ]:
def Index_1(delta, gamma, EMG):
  index_1 = np.array([])
  for i in range(len(delta)):
    value = (EMG[i]*gamma[i])/(delta[i])
    index_1 = np.append(index_1, [value])
  return index_1
  
def Index_2(delta, theta, sigma):
  index_2 = np.array([])
  for i in range(len(delta)):
    value = (sigma[i]*delta[i])/(theta[i])
    index_2 = np.append(index_2, [value])
  return index_2
  
def Index_3(delta, theta, gamma):
  index_3 = np.array([])
  for i in range(len(delta)):
    value = (theta[i]*gamma[i])/(delta[i])
    index_3 = np.append(index_3, [value])
  return index_3
  
def Index_4(delta, theta):
  index_4 = np.array([])
  for i in range(len(delta)):
    value = delta[i]/theta[i]
    index_4 = np.append(index_4, [value])
  return index_4
  
def index_W(theta, gamma, EMG):
  index_w = np.array([])
  for i in range(len(theta)):
    value = EMG[i]*EMG[i]*((gamma[i])/(theta[i]))
    index_w = np.append(index_w, [value])
  return index_w

def index_R(delta, theta, sigma, EMG, gamma):
  index_r = np.array([])
  for i in range(len(delta)):
    value = ((theta[i]*2)*gamma[i])/(delta[i]*delta[i]*EMG[i]**2)
    index_r = np.append(index_r, [value])
  return index_r

def index_N(delta, sigma, gamma):
  index_n = np.array([])
  for i in range(len(delta)):
    value = (sigma[i]*delta[i])/(gamma[i]**2)
    index_n = np.append(index_n, [value])
  return index_n

In [ ]:
def psd_multitaper(lfp_data, fs, frequency_band, window_length):
    all_power_sum = []
    
    # loop through each segment
    for start in range(0, len(lfp_data) - window_length + 1, window_length):
        window = lfp_data[start:min(start + window_length, len(lfp_data))]

        # compute power spectral density using multitaper method
        psd, freqs = mne.time_frequency.psd_array_multitaper(window, fs, fmin=frequency_band[0], fmax=frequency_band[1], n_jobs=1, verbose = 'warning')

        # compute total power within frequency band
        freq_indices = (freqs >= frequency_band[0]) & (freqs <= frequency_band[1])
        curr_sum = np.sum(psd)
        all_power_sum.append(curr_sum)

    return all_power_sum

In [ ]:
def wei_normalizing(data):
    data = np.array(data)

    bottom = data[data <= np.nanpercentile(data, 10, axis=0)]
    top = data[data >= np.nanpercentile(data, 90, axis=0)]

    bottom_avg = np.average(bottom) if len(bottom) > 0 else 0
    top_avg = np.average(top) if len(top) > 0 else 1

    denom = top_avg - bottom_avg if top_avg != bottom_avg else 1
    normalized_data = (data - bottom_avg) / denom
    normalized_data = np.clip(normalized_data, 0.05, 1)

    return normalized_data

In [ ]:
def fragment_join(subject, night, type):
    base = f"D:/EEG_Data_stage/{subject}/iEEG/{type}"
    score_base = f"D:/EEG_Data_stage/{subject}/iEEG/U_sleep_API_10s"
    files = []
    scores = []
    for file in os.listdir(base):
        if f"night{night}" in file and ".vhdr" in file:
            files.append(os.path.join(base,file))
            scores.append(os.path.join(score_base,file.replace(".vhdr", "_hypnogram.npy")))
    return files, scores        

In [ ]:
def create_plots(raw, hpc_channel, pfc_channel, states, output_dir, epoch_length, fs, window_length):
    # Bandpass EMG
    raw.filter(l_freq=10, h_freq=70, picks='EMG1-EMG2')
    hpc_data = raw.get_data(picks=hpc_channel)[0]
    hpc_tag = hpc_channel
    raw_hpc = np.ravel(hpc_data)
    pfc_data = raw.get_data(picks=pfc_channel)[0]
    pfc_tag = pfc_channel
    raw_pfc = np.ravel(pfc_data)
    EMG = raw.get_data(picks='EMG1-EMG2')[0]
    EMG = EMG[:len(EMG)// (epoch_length * fs) * (epoch_length * fs)] 
    EMG = EMG.reshape(-1, (epoch_length * fs))
    EMG = EMG.sum(axis=1)
    EMG = abs(hilbert(EMG))
    amp_thresh = [6,4]       
    time_win_thresh = [0.2,0.1]
    raw_hpc = A.removeArtefacts(raw_hpc, 250, amp_thresh, time_win_thresh)[0]
    raw_pfc = A.removeArtefacts(raw_pfc, 250, amp_thresh, time_win_thresh)[0]
    
    sleep_scoring = np.ravel(states)
    score_labels = {0: 'Wake', 1: 'N1', 2:'N2', 3: 'N3', 4: 'REM'}
    mapped_scores = np.array(sleep_scoring)
    sleep_time_minutes = np.arange(0, len(sleep_scoring)) * epoch_length / 60
    num_labels = {label: num for num, label in enumerate(score_labels.values(), start=1)}
    # Get EOG
    EOG1, EOG2 = np.ravel(raw.get_data(picks='EOG1')[0]), np.ravel(raw.get_data(picks='EOG2')[0])
    epochs = np.arange(len(mapped_scores))
    # Establshing hypno epochs here prevents index mismatches later
    # --- Set up cache directory for PSD computation ---
    cache_dir = os.path.join(output_dir, "cached_psd_rodent")
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"subject_{subject}_night_{night}_psd.pkl")
    
    # --- Try loading cached PSDs to save computation time ---
    if os.path.exists(cache_file):
        print(f"✅ Loading cached PSDs for subject {subject}, night {night}...")
        with open(cache_file, "rb") as f:
            noise, delta, theta, sigma, beta, gamma, alpha = pickle.load(f)
    else:
        print(f"⚙️ Computing PSDs for subject {subject}, night {night}...")
        
        # List of signals to compute PSDs for each band
        raws = [raw_pfc, raw_pfc, raw_hpc, raw_pfc, raw_pfc, raw_pfc, raw_hpc]
        fr_bands = [noise_band, delta_band, theta_band, sigma_band, beta_band, gamma_band, alpha_band]
        
        # Compute PSDs in parallel
        noise, delta, theta, sigma, beta, gamma, alpha = Parallel(n_jobs=7)(
            delayed(psd_multitaper)(raw_sig, fs, band, window_length)
            for raw_sig, band in zip(raws, fr_bands)
        )
        
        # Save PSDs to cache
        with open(cache_file, "wb") as f:
            pickle.dump((noise, delta, theta, sigma, beta, gamma, alpha), f)
        print(f"💾 PSDs saved to cache: {cache_file}")
    
    noise_norm = wei_normalizing(noise)
    delta_norm = wei_normalizing(delta)
    theta_norm = wei_normalizing(theta)
    sigma_norm = wei_normalizing(sigma)
    beta_norm = wei_normalizing(beta)
    gamma_norm = wei_normalizing(gamma)
    alpha_norm = wei_normalizing(alpha)
    #total_norm = wei_normalizing(total)
    EMG_norm = wei_normalizing(EMG)
    noise_smoothed = np.convolve(np.convolve(np.convolve(noise_norm, np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same')
    theta_smoothed = np.convolve(np.convolve(np.convolve(theta_norm, np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same')
    delta_smoothed = np.convolve(np.convolve(np.convolve(delta_norm, np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same')
    alpha_smoothed = np.convolve(np.convolve(np.convolve(alpha_norm, np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same')
    beta_smoothed = np.convolve(np.convolve(np.convolve(beta_norm, np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same')
    sigma_smoothed = np.convolve(np.convolve(np.convolve(sigma_norm, np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same')
    gamma_smoothed = np.convolve(np.convolve(np.convolve(gamma_norm, np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same'), np.ones(5)/5, mode='same')
    EMG = np.convolve(EMG, np.ones(20*fs)/(20*fs), mode='same')
    # Index calculations
    index_n = index_N(delta_norm, sigma_norm, gamma_norm)
    index_r = index_R(delta_norm, theta_norm, sigma_norm, EMG_norm, gamma_norm)
    index_w = index_W(theta_norm, gamma_norm, EMG_norm)
    index_1 = Index_1(delta_norm, gamma_norm, EMG_norm)
    index_2 = Index_2(delta_norm, theta_norm, sigma_norm)
    index_3 = Index_3(delta_norm, theta_norm, gamma_norm)
    index_4 = Index_4(delta_norm, theta_norm)
    
    min_len = min(len(epochs), len(index_w), len(index_n), len(index_r))
    epochs = epochs[:min_len]
    hypno_epochs = epochs
    mapped_scores = mapped_scores[:min_len]
    sleep_scoring = sleep_scoring[:min_len]
    index_w = index_w[:min_len]
    index_n = index_n[:min_len]
    index_r = index_r[:min_len]
    
    F.combined_indices_figure(epochs, epochs, mapped_scores, [index_w, index_r, index_n], ["Index W", "Index R", "Index N"], output_dir, ('A', 'B', 'C'))
    # # Powerband plot
    # F.normalised_powers(EMG_norm, noise_norm, delta_norm, theta_norm, sigma_norm, beta_norm, gamma_norm, alpha_norm, hypno_epochs, sleep_scoring, output_dir)
    # # Normalised EMG
    # F.normalised_EMG(EMG_norm, output_dir)
    # # Raw signals
    # F.raw_signals(states, raw_hpc, raw_pfc, pfc_tag, hpc_tag, output_dir, fs, epoch_length)
    # # Indices vs a hypnogram
    # F.indices_vs_hypnogram(epochs, hypno_epochs, index_w, index_n, index_r, mapped_scores, output_dir)
    # # Index barplot
    # F.index_barplot(index_n, index_r, index_w, mapped_scores, output_dir)
    # # index PCA
    # F.index_pca([index_w, index_n, index_r, index_1, index_2, index_3, index_4], mapped_scores, output_dir)
                

### Establish parameters

In [ ]:
# Intra-cranial
electrodes = {"2":["TL05", "TLR06"],
              "7":["TL03", "TLR06"],
              "15":["TL05", "TLR01"], 
              "28":["TL07", "TLR06"], 
              "31":["TL05", "TLR12"], 
              "87":["TL05", "TLR01"]}

In [ ]:
# # Extra-cranial
# electrodes = {
#     "2": ["Oz-Cz", "C3-Cz"],
#     "7": ["Oz-Cz", "C3-Cz"],
#     "15": ["Oz-Cz", "C3-Cz"], 
#     "28": ["Oz-Cz", "C3-Cz"], 
#     "31": ["Oz-Cz", "C3-Cz"],
#     "63": ["Oz-Cz", "C3-Cz"],
#     "67": ["Oz-Cz", "C3-Cz"], 
#     "69": ["Oz-Cz", "C3-Cz"], 
#     "84": ["Oz-Cz", "C3-Cz"], 
#     "85": ["Oz-Cz", "C3-Cz"],
#     "86": ["Oz-Cz", "C3-Cz"],
#     "87": ["C4-Cz", "C3-Cz"],
#     "132": ["C4-Cz", "C3-Cz"], 
#     "134": ["F3", "C4"], 
#     "135": ["F3", "C4"]
# }

In [ ]:
fs = 250
epoch_length = 10 
window_length = epoch_length * fs
base_dir = "D:/EEG_Data_stage"
processed_log = os.path.join(base_dir, "processed_subjects_rodent_v2.txt")
file_type = "converted_intra_upscale_strips"

# Load already processed subjects/nights
# Prevents rerunning the process for already processed subjects/nights
if os.path.exists(processed_log):
    with open(processed_log, "r") as f:
        processed = set(line.strip() for line in f if line.strip())
else:
    processed = set()

for file in os.listdir(base_dir):
    if file =="15":
        print(f"-----------------------------------Converting subject: {file}-----------------------------------")
        channels = electrodes[file] 
        
                # Handle "28" special case
        nights = [1, 2]

        # Sometimes nights might be missing, fix that here
        if file in ["28", "87"]:
            if file == "28":
                nights = [2]
            else:
                nights = [1]
        for night in nights:
            if file == "31" and night == 2:
                continue
            subject_night = f"{file}_night{night}"

            # Skip if already processed
            if subject_night in processed:
                print(f"Skipping {subject_night} — already processed.")
                continue

            print(f"Converting night: {night}")
            subject = file
            night_str = f"{night}"

            hpc_channel = channels[0]
            pfc_channel = channels[1]
            output_dir = os.path.join(base_dir, "plots_rodent", file_type, subject, f"night{night_str}")
            os.makedirs(output_dir, exist_ok=True)

            files, scores_files = fragment_join(subject, night_str, file_type)
            raw_list = [mne.io.read_raw_brainvision(f, preload=True) for f in files]
            raw = mne.concatenate_raws(raw_list)
            score_list = [np.load(f) for f in scores_files]
            states = np.concatenate(score_list)

            create_plots(raw, hpc_channel, pfc_channel, states, output_dir, epoch_length, fs, window_length)

            # Log successful processing
            with open(processed_log, "a") as f:
                f.write(subject_night + "\n")
            processed.add(subject_night)

            print(f"-----------------------------------Finished creating plots for subject: {subject}; night {night}-----------------------------------")
        # except IndexError as e:
        #     print(f"Night {night} is out of range. Skipping {subject_night}.")
        #     print(f"Error: {e}")
